In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import json
import shutil

In [2]:
DATA_DIR = Path("/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/LABEL")
PHOTO_DIR = Path("/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/PHOTO")

MATCH_DIR_LABEL_REGEX = re.compile('TL_3.상추[0-9]+')
MATCH_DIR_PHOTO_REGEX = re.compile('TS_3.상추[0-9]+')

REF_PHOTO_DIR = Path("/Users/mainframe/Workspace/Graduate/dataset/photos")
REF_LABEL_DIR = Path("/Users/mainframe/Workspace/Graduate/dataset/labels")

MIN_DIR = 54
MAX_DIR = 75

In [3]:
list_of_photos = []

#Move Every Photos that has good portrait.
for entry in PHOTO_DIR.iterdir():
    if entry.is_dir() and MATCH_DIR_PHOTO_REGEX.match(entry.name):
        dir_no = int(entry.name[-2] + entry.name[-1])
        if MIN_DIR <= dir_no and dir_no <= MAX_DIR:
            for subentry in entry.iterdir():
                if subentry.is_file():
                    list_of_photos.append(subentry)
                    shutil.copy(subentry, REF_PHOTO_DIR / subentry.name)

In [5]:
#Perform and exhuastive search, and yes this is intentional.
for entry in DATA_DIR.iterdir():
    if entry.is_dir() and MATCH_DIR_LABEL_REGEX.match(entry.name):
        for subentry in entry.iterdir():
            if subentry.is_file():
                #print(f"Moving {subentry} to {REF_PHOTO_DIR / subentry.name}")
                shutil.copy(subentry, REF_LABEL_DIR / subentry.name)

In [6]:
list_of_photos[0:5]

[PosixPath('/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/PHOTO/TS_3.상추62/C30_L01_06_003_756540.jpg'),
 PosixPath('/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/PHOTO/TS_3.상추62/C30_L01_06_001_778768.jpg'),
 PosixPath('/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/PHOTO/TS_3.상추62/C30_L01_06_003_756554.jpg'),
 PosixPath('/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/PHOTO/TS_3.상추62/C30_L01_06_001_855585.jpg'),
 PosixPath('/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/PHOTO/TS_3.상추62/C30_L01_06_001_806877.jpg')]

In [7]:
def safe_avg_of(data, column_name):
    column = [e[column_name] for e in data]
    if None in column:
        return None
    num_column = [float(n) for n in column]
    return sum(num_column) / len(num_column)

In [8]:
def growth_to_number(growth):
    if growth == '정식기':
        return 1
    elif growth == '생육기':
        return 2
    elif growth == '수확기':
        return 3
    raise ValueError("Some other value has been detected")

In [9]:
def parse_json_file(file_path):
    """Reads a single JSON file and returns a dictionary or None if failed."""
    try:
        # Using pathlib's read_text handles the file opening/closing for you
        #dir_no, file_path = file_info
        raw_data = file_path.read_text(encoding='utf-8')
        
        data = json.loads(raw_data)

        leaf_ids = []
        stem_ids = []

        for cat in data["categories"]:
            if cat["name"] == "잎":
                leaf_ids.append(cat["id"])
            else:
                stem_ids.append(cat["id"])

        environment = data["envrionments"]
        
        extracted = {
            #'dir_no': dir_no,
            'file_name': file_path.name.split(".")[0],
            'growth': growth_to_number(data["images"]["growth_stage"]),
            'captured': data["images"]["date_captured"],
            'kind': data["images"]["kind_type"],
            'temp': safe_avg_of(environment, "ti_value"),
            'humid': safe_avg_of(environment, "hi_value"),
            'co2': safe_avg_of(environment, "ci_value"),
            'light': safe_avg_of(environment, "ir_value"),
            'hyd_temp': safe_avg_of(environment, "tl_value"),
            'hyd_ec': safe_avg_of(environment, "ei_value"),
            'hyd_ph': safe_avg_of(environment, "pl_value"),
            'stem_area': sum([box["area"] if box["id"] in stem_ids else 0 for box in data["annotations"]]),
            'leaf_area': sum([box["area"] if box["id"] in leaf_ids else 0 for box in data["annotations"]]),
        }
        
        return extracted 
    except (json.JSONDecodeError, IOError) as e:
        print(f"Error parsing {file_path.name}: {e}")
    except TypeError as e:
        print(f"Error parsing {file_path.name}: {e}")
        return None

In [12]:
parsed_records = [parse_json_file(REF_LABEL_DIR / f"{photo_path.name.split(".")[0]}.json") for photo_path in list_of_photos]
df = pd.DataFrame(parsed_records)

In [13]:
df.head()

,file_name,growth,captured,kind,temp,humid,co2,light,hyd_temp,hyd_ec,hyd_ph,stem_area,leaf_area
0,C30_L01_06_003_756540,2,2022-01-08 02:16:24,로메인,20.0,81.0,267.5,79.0,20.70,0.05,7.4,273729.24,234047.59
1,C30_L01_06_001_778768,2,2022-01-11 16:16:24,로메인,18.0,82.5,369.5,0.0,19.90,0.05,NaN,325292.96,297204.99
2,C30_L01_06_003_756554,2,2022-01-08 06:16:24,로메인,19.0,78.5,247.0,87.0,20.65,0.05,7.5,282746.62,240854.46
3,C30_L01_06_001_855585,2,2022-01-21 22:16:24,로메인,19.0,74.5,1096.5,67.0,19.80,0.04,7.0,861586.26,334361.66
4,C30_L01_06_001_806877,2,2022-01-15 16:16:24,로메인,18.0,85.5,323.5,0.0,19.50,0.04,7.5,477949.56,317834.15


In [14]:
output_file = Path("plant_gotchi_dataset.parquet")

# We use 'pyarrow' as the engine and 'snappy' for fast compression
df.to_parquet(
    output_file, 
    engine='pyarrow', 
    compression='snappy',
    index=False  # Usually, we don't want the pandas row index in our data
)

print(f"Dataset successfully saved to: {output_file.absolute()}")

Dataset successfully saved to: /Users/mainframe/Workspace/Graduate/notebooks/plant_gotchi_dataset.parquet
